# run_modeling: train the CatBoost classifier

1. BAR-ATH: load the tracking and, for a check only, the published 5 minutes of ground truth; preprocess; compute
   features.
2. Measure how BAR-ATH's tracking is broken (no labels needed).
3. Load the external matches (7 DFL / IDSSE and 3 Metrica sample games), damage them the same way and compute the
   same features.
4. Train CatBoost on eight matches with early stopping on the other two (J03WR9 and METRICA3), fit a temperature,
   then refit on all ten.
5. Save the model bundle so `run_decoder.ipynb` can load the trained model.
6. Check the classifier alone (per-frame argmax, no decoder) on the published 5 minutes.

## Load

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent   # the repository root: notebooks run from possession/
PIPELINE_DIR = REPO_ROOT / "possession"
ARTIFACT_DIR = PIPELINE_DIR / "artifacts"
OUTPUT_DIR = PIPELINE_DIR / "outputs"


def load_notebook_defs(filename: str, namespace: dict) -> None:
    loaded = namespace.setdefault("_LOADED_NOTEBOOKS", set())
    if filename in loaded:
        return
    loaded.add(filename)
    nb = json.loads((PIPELINE_DIR / filename).read_text(encoding="utf-8"))
    for i, cell in enumerate(nb["cells"]):
        if cell["cell_type"] == "code" and "defs" in cell.get("metadata", {}).get("tags", []):
            exec(compile("".join(cell["source"]), f"{filename}::cell{i}", "exec"), namespace)

In [ ]:
load_notebook_defs("helper_data.ipynb", globals())        # DFL + Metrica loading, degradation, feature sets
load_notebook_defs("helper_decoder.ipynb", globals())     # metrics + label priors (decoder itself runs in run_decoder)
load_notebook_defs("helper_reporting.ipynb", globals())   # evaluation on any ground truth

## Configuration

In [ ]:
# ---- external training data: 7 DFL + 3 Metrica (no synthetic data) ----
EXTERNAL_MATCHES = list(IDSSE_MATCHES) + list(METRICA_MATCHES)
# train / validation split: hold out one DFL match and one NATIVE (Metrica) match, so validation reflects both domains
VAL_MATCHES = ["J03WR9", "METRICA3"]
ALLOW_DOWNLOAD = True
N_AUG = 2                                 # independently degraded copies of every training match
TRAIN_STRIDE = 5                          # keep every 5th frame for training (neighbouring frames are near-duplicates)
REFIT_ALL = True                          # after fitting, retrain on all matches with the chosen number of trees

# ---- check: the published 5-minute ground truth (scoring only) ----
GT_PATH = REPO_ROOT / "evaluation" / "gt" / "BAR-ATH_possession_gt_public.csv"   # the published 5 minutes: a check, never a choice
CHUNK_S = 300.0
FORCE_REBUILD = False

# ---- the saved model bundle (loaded back in run_decoder.ipynb) ----
MODEL_NAME = "possession_v1"
PROBS_CACHE = ARTIFACT_DIR / f"{MODEL_NAME}_target_probs.npz"   # target probabilities for run_decoder (optional convenience)

PRE_CFG = PreprocessConfig()
FEAT_CFG = FeatureConfig()
print("train matches:", [m for m in EXTERNAL_MATCHES if m not in VAL_MATCHES])
print("val matches  :", VAL_MATCHES, "(one DFL + one native Metrica)")

## Step 1: BAR-ATH: load, preprocess, features

In [ ]:
t_all = time.time()
inputs = load_match_inputs(GT_PATH)
clean = load_or_build_clean(inputs.raw, PRE_CFG, force=FORCE_REBUILD)
T = clean.n_frames
graded = clean.half_id() > 0
X_target = WindowFeatureBuilder().transform(SpatialFeatureExtractor(FEAT_CFG).tabular(clean), clean.half_blocks)
gt = inputs.gt
gblocks = gt_blocks(gt, clean.half_blocks)
print(f"frames={T} | graded={int(graded.sum())} | ground-truth frames={int((gt != 'X').sum())} | model features={X_target.shape[1]}")

## Step 2: measure how broken BAR-ATH's tracking is

In [ ]:
profile = DegradationProfile.from_target(inputs.raw)
print(profile.summary())

## Step 3: external matches, damaged like BAR-ATH
The first run parses the 7 DFL files (large XML) and the 3 Metrica games and caches them in `artifacts/external/`.

In [ ]:
t0 = time.time()
train_ids = [m_ for m_ in EXTERNAL_MATCHES if m_ not in VAL_MATCHES]
ext = build_external_sets(train_ids, VAL_MATCHES, profile, PRE_CFG, FEAT_CFG, n_aug=N_AUG, stride=TRAIN_STRIDE,
                          allow_download=ALLOW_DOWNLOAD)
assert list(ext.X_train.columns) == list(X_target.columns), "external and target feature columns differ"
n_native = sum(is_native_match(m) for m in train_ids)
print(f"external training rows: {len(ext.y_train):,} " +
      str({k: int((ext.y_train == v).sum()) for k, v in LABEL_TO_ID.items()}) +
      f" | train matches {len(train_ids)} ({n_native} native + {len(train_ids) - n_native} DFL) | built in {time.time() - t0:.0f}s")

## Step 4: possession priors from the external labels
The decoder needs transition priors from the labelled training matches. We compute them here and store them in the
bundle, so `run_decoder.ipynb` does not need the training data.

In [ ]:
priors = label_priors([ids_to_labels(ext.labels[m_][0]) for m_ in train_ids], [ext.labels[m_][1] for m_ in train_ids])
gt_priors = label_priors([gt], [gblocks])
print(pd.DataFrame({"external (training matches)": priors, "ground truth": gt_priors}).round(3).to_string())

## Step 5: train CatBoost on the external matches
Early stopping and the temperature both use the two validation matches (J03WR9 and METRICA3). With `REFIT_ALL` the
model is then refitted on all ten matches with 15% more trees than the early-stopping point.

In [ ]:
t0 = time.time()
clf_ext = PossessionClassifier().fit(ext.X_train, ext.y_train, eval_set=ext.val_rows(TRAIN_STRIDE))
print(f"trained on {len(ext.y_train):,} external frames in {time.time() - t0:.0f}s | best iteration {clf_ext.best_iteration_}")

val_logits = {m_: clf_ext.predict_logits(v.X) for m_, v in ext.val.items()}
y_val = np.concatenate([ext.val[m_].y for m_ in VAL_MATCHES])
temp_ext = fit_temperature(np.concatenate([val_logits[m_] for m_ in VAL_MATCHES]), y_val)
val_probs = {m_: softmax(val_logits[m_] / temp_ext) for m_ in VAL_MATCHES}
print(f"temperature {temp_ext:.2f} | external validation, per frame:",
      frame_scores(np.concatenate([val_probs[m_] for m_ in VAL_MATCHES]), y_val))

In [ ]:
if REFIT_ALL:                                            # same recipe, all matches, fixed tree count
    t0 = time.time()
    n_trees = int((clf_ext.best_iteration_ + 1) * 1.15)
    clf_ext = PossessionClassifier().fit(*ext.all_rows(TRAIN_STRIDE), iterations=n_trees)
    print(f"refit on all {len(EXTERNAL_MATCHES)} matches with {n_trees} trees in {time.time() - t0:.0f}s")
clf_ext.temperature = temp_ext

## Step 6: save the model bundle
The bundle holds the CatBoost model, feature columns, temperature, decoder priors and the feature and preprocessing
settings. It also stores a default decoder built from the priors; `run_decoder.ipynb` replaces it with the chosen
setting and saves the result under `artifacts/models/`.

In [ ]:
HMM_BASE = HMMConfig(mean_possession_s=priors["mean_possession_s"], mean_dead_s=priors["mean_dead_s"],
                     handover_share=priors["handover_share"], enable_x=False)
save_bundle(MODEL_NAME, clf_ext, HMM_BASE, FEAT_CFG, PRE_CFG,
            info={"external_matches": EXTERNAL_MATCHES, "val_matches": VAL_MATCHES, "n_aug": N_AUG,
                  "train_stride": TRAIN_STRIDE, "refit_all": REFIT_ALL, "decoder_tuned": False,
                  "native_matches": list(METRICA_MATCHES), "gt_path": str(GT_PATH)})
print(f"model saved in {MODEL_DIR / MODEL_NAME}")

# cache the target probabilities so run_decoder can skip re-running the model if desired
probs_target = clf_ext.predict_proba(X_target, rows=np.flatnonzero(graded))
ARTIFACT_DIR.mkdir(exist_ok=True)
np.savez_compressed(PROBS_CACHE, probs=probs_target)
print(f"target probabilities cached -> {PROBS_CACHE.name}")

## Step 7: score the classifier alone
No decoder yet: the labels are the per-frame argmax of the calibrated probabilities. `run_decoder.ipynb` adds the HMM
decoder and scores again.

In [ ]:
arg_target = argmax_labels(probs_target, clean.half_blocks)
res = evaluate_on_gt(arg_target, probs_target, gt, clean, name="classifier (argmax)", chunk_s=CHUNK_S, plots=False)
print(f"\n[run_modeling] done in {time.time() - t_all:.0f}s - model saved as '{MODEL_NAME}', ready for run_decoder.ipynb")

## Step 8: plots of the classifier labels (confusion matrix, predicted against true counts)

In [ ]:
plot_prediction_summary(gt, arg_target, clean.half_blocks, probs=probs_target, ball_src=clean.ball_src,
                        chunk_s=0, title="Classifier (per-frame argmax) - BAR-ATH",
                        path=FIG_DIR / "run_modeling_prediction.png")